# Machine Learning for CICY 4-Folds

Following the analysis in [arXiv:2007.13379](https://arxiv.org/abs/2007.13379) and [arXiv:2007.15706](https://arxiv.org/abs/2007.15706), we apply similar concepts to CICY 4-folds.
The idea is to see whether the Inception network can also be applied in higher dimensions.

## Feature Engineering

In this notebook we first build some engineered features providing additional information with respect to the original dataset.

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from mltools import *

ctx = Context(img_dir='img', log_dir='log', dat_dir='data', session='feat_eng', subdir=False)
log = ctx.logger()

log.info('Feature Engineering.')

## Download and Read the Dataset

The dataset was first introduced in [arXiv:1405.2073](http://arxiv.org/abs/1405.2073).
The authors provide Hodge numbers and other characteristic invariants of CICY 4-folds.

In [2]:
tab                = Table('https://www.lpthe.jussieu.fr/~erbin/files/data/cicy4.json.gz', ctx=ctx).read(orient='index')
df, (n_rows, n_cols) = tab.data()
log.debug(f'Dataset: {n_rows:d} rows X {n_cols:d} columns')

No. of rows:    921497.
No. of columns: 9.


In [3]:
df.eda.full_info()

No. of rows:    921497
No. of columns: 9
Index:          <class 'pandas.core.indexes.numeric.Int64Index'>
RAM usage:      246.84 MB.




,dtypes,NA cases,mean,std,0% (min),1%,10%,25%,50% (median),75%,90%,99%,100% (max)
euler,int64,0,334.974098,86.830941,0.0,0.0,288.0,288.0,318.0,360.0,420.0,636.0,2610.0
favour,bool,0,0.535587,0.498732,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
h11,float64,15813,10.065124,2.424114,1.0,5.0,7.0,8.0,10.0,12.0,13.0,16.0,24.0
h21,float64,15813,0.816733,1.882068,0.0,0.0,0.0,0.0,0.0,1.0,3.0,9.0,33.0
h22,float64,15813,240.829490,49.605669,204.0,204.0,204.0,212.0,226.0,252.0,294.0,440.0,1752.0
h31,float64,15813,39.550615,13.719057,20.0,25.0,28.0,31.0,36.0,43.0,55.0,92.0,426.0
isprod,bool,0,0.017160,0.129868,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0
matrix,object,0,--,--,--,--,--,--,--,--,--,--,--
size,object,0,--,--,--,--,--,--,--,--,--,--,--


## Feature Engineering

Notice that each configuration matrix is a $m \times k$ table
\begin{equation}
    X =
    \begin{bmatrix}
        \mathbb{P}^{n_1}\colon & a_1^1 & \dots & a_k^1 \\
        \mathbb{P}^{n_2}\colon & a_1^2 & \dots & a_k^2 \\
        \vdots & & & \\
        \mathbb{P}^{n_m}\colon & a_1^m & \dots & a_k^m \\
    \end{bmatrix}
\end{equation}
and
\begin{equation}
    n_r = \sum\limits_{\alpha = 1}^k\, a_{\alpha}^r - 1,
    \qquad
    r = 1, 2, \dots, m.
\end{equation}

We then compute the number of projective spaces $m$ (i.e. the number of rows), the number of equations $k$ (i.e. the number of columns), the number of $\mathbb{P}^1$ $f$ (i.e. the number of rows such that $n_r = 1$ $\forall r = 1, 2, \dots, m$), the number of $\mathbb{P}^2$ (i.e. the number of rows such that $n_r = 2$ $\forall r = 1, 2, \dots, m$), and the number $F$ of $\mathbb{P}^{n_r}$ such that $n_r \neq 1$.

The excess number is then computed as
\begin{equation}
    N_{ex} = \sum\limits_{r = 1}^F\, n_r + f + m - 2 k.
\end{equation}

The norm of the matrix is the Frobenius norm $\left|\left| A \right|\right| = \sqrt{\sum\limits_{r = 1}^m \sum\limits_{\alpha = 1}^k\, \left| a_{\alpha}^r \right|^2}$.

Other engineered features include the list of the dimensions of the projective spaces and the list of the degrees of the polynomials.

In [4]:
# no. of projective spaces (rows)
log.debug('Computing no. of projective spaces (rows)...')
df['num_cp']         = df['size'].apply(lambda s: s[0]).astype(np.int)

# no. of equations (columns)
log.debug('Computing no. of equations (columns)...')
df['num_eqs']        = df['size'].apply(lambda s: s[1]).astype(np.int)

# no. of P^1
log.debug('Computing no. of P^1...')
df['num_cp_1']       = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum((v == 1).astype(np.int)))

# no. of P^2
log.debug('Computing no. of P^2...')
df['num_cp_2']       = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum((v == 2).astype(np.int)))

# no. of P^n with n != 1
log.debug('Computing no. of P^n with n != 1...')
df['num_cp_neq1']    = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum((v > 1).astype(np.int)))

# excess number
log.debug('Computing excess number...')
df['num_ex']         = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum(v[(v > 1)])) \
                       + df['num_cp_1'] + df['num_cp'] - 2 * df['num_eqs']

# Frobenius norm and rank of the matrix
log.debug('Computing Frobenius norm and rank of the matrix...')
df['norm_matrix']    = df['matrix'].apply(np.linalg.norm)
df['rank_matrix']    = df['matrix'].apply(np.linalg.matrix_rank)

# list of the dimensions of the projective spaces
log.debug('Computing list of the dimensions of the projective spaces...')
df['dim_cp']         = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1)
df['min_dim_cp']     = df['dim_cp'].apply(np.min)
df['max_dim_cp']     = df['dim_cp'].apply(np.max)
df['mean_dim_cp']    = df['dim_cp'].apply(np.mean)
df['std_dim_cp']     = df['dim_cp'].apply(np.std)
df['median_dim_cp']  = df['dim_cp'].apply(np.median)
df['quart_dim_cp']   = df['dim_cp'].apply(lambda l: np.quantile(l, [0.25, 0.75]))

# list of the degrees of the equations
log.debug('Computing list of the degrees of the equations...')
df['deg_eqs']        = df['matrix'].apply(lambda m: np.max(m, axis=1))
df['min_deg_eqs']    = df['deg_eqs'].apply(np.min)
df['max_deg_eqs']    = df['deg_eqs'].apply(np.max)
df['mean_deg_eqs']   = df['deg_eqs'].apply(np.mean)
df['std_deg_eqs']    = df['deg_eqs'].apply(np.std)
df['median_deg_eqs'] = df['deg_eqs'].apply(np.median)
df['quart_deg_eqs']  = df['deg_eqs'].apply(lambda l: np.quantile(l, [0.25, 0.75]))

## Save the Dataset

We then save the dataset to file.

In [5]:
df.eda.full_info()

No. of rows:    921497
No. of columns: 31
Index:          <class 'pandas.core.indexes.numeric.Int64Index'>
RAM usage:      893.16 MB.




,dtypes,NA cases,mean,std,0% (min),1%,10%,25%,50% (median),75%,90%,99%,100% (max)
euler,int64,0,334.974098,86.830941,0.000000,0.000000,288.000000,288.000000,318.000000,360.000000,420.000000,636.000000,2610.000000
favour,bool,0,0.535587,0.498732,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
h11,float64,15813,10.065124,2.424114,1.000000,5.000000,7.000000,8.000000,10.000000,12.000000,13.000000,16.000000,24.000000
h21,float64,15813,0.816733,1.882068,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,3.000000,9.000000,33.000000
h22,float64,15813,240.829490,49.605669,204.000000,204.000000,204.000000,212.000000,226.000000,252.000000,294.000000,440.000000,1752.000000
h31,float64,15813,39.550615,13.719057,20.000000,25.000000,28.000000,31.000000,36.000000,43.000000,55.000000,92.000000,426.000000
isprod,bool,0,0.017160,0.129868,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
matrix,object,0,--,--,--,--,--,--,--,--,--,--,--
size,object,0,--,--,--,--,--,--,--,--,--,--,--
num_cp,int64,0,8.966027,1.782741,1.000000,5.000000,7.000000,8.000000,9.000000,10.000000,11.000000,13.000000,16.000000


In [6]:
# write the file to file
tab = tab.update(df)
tab.write('cicy4_eng.json.gz', orient='index')
log.info('Dataset saved to file.')

## Matrix Only

We also build a dataset using just the matrix for later convenience.

In [7]:
mat_tab = Table('https://www.lpthe.jussieu.fr/~erbin/files/data/cicy4.json.gz', ctx=ctx).read(orient='index')
matrix  = mat_tab.get_df()
matrix = matrix.eda.complete_cases
log.info('Creating matrix only database...')

We then determine the size of the largest matrix:

In [8]:
size = matrix['matrix'].apply(lambda x: np.shape(x)).max()
print(f'Largest matrix: {size[0]:d} rows X {size[1]:d} columns.')

Largest matrix: 16 rows X 20 columns.


We finally select the matrix column and the Hodge numbers.
We then pad the matrix column into the shape of the largest matrix.

In [9]:
matrix = matrix[['matrix', 'h11', 'h21', 'h31', 'h22']]
matrix = matrix.eda.pad('matrix', size)
matrix = matrix.eda.convert_dtype(columns=['h11', 'h21', 'h31', 'h22'], dtype=np.int)

Eventually we save the dataset:

In [10]:
mat_tab = mat_tab.update(matrix)
mat_tab.write('cicy4_matrix.json.gz', orient='index')
log.info('Dataset saved to file.')